# preparing Family Column

In [ ]:
import pandas as pd
import re
import sys
from pathlib import Path
import numpy as np
import pandas as pd

In [ ]:
BASE_DIR = Path.cwd().parent

STAGE1_TRAIN_PATH = (
    BASE_DIR
    / "data"
    / "processed"
    / "stage1"
    / "stage1_train_models.csv"
)
STAGE1_VAL_PATH = (
    BASE_DIR
    / "data"
    / "processed"
    / "stage1"
    / "stage1_val_models.csv"
)
print("Current directory:", Path.cwd())
print("Stage 1 train file:", STAGE1_TRAIN_PATH.resolve())
print("Stage 1 val file:", STAGE1_VAL_PATH.resolve())

#STAGE2_PATH = (
#    BASE_DIR
#    / "data"
#    / "processed"
#    / "stage2"
#    / "stage2_df.csv"
#)

print("Current directory:", Path.cwd())
print("Stage 1 train file:", STAGE1_TRAIN_PATH.resolve())
print("Stage 1 val file:", STAGE1_VAL_PATH.resolve())

print("File TRAIN exists:", STAGE1_TRAIN_PATH.exists())
print("File VAL exists:", STAGE1_VAL_PATH.exists())

Current directory: e:\DEPI-Grad\DEPI-GRADUATION-PROJECT\notebooks
Stage 1 train file: E:\DEPI-Grad\DEPI-GRADUATION-PROJECT\data\processed\stage1\stage1_train_models.csv
Stage 1 val file: E:\DEPI-Grad\DEPI-GRADUATION-PROJECT\data\processed\stage1\stage1_val_models.csv
Current directory: e:\DEPI-Grad\DEPI-GRADUATION-PROJECT\notebooks
Stage 1 train file: E:\DEPI-Grad\DEPI-GRADUATION-PROJECT\data\processed\stage1\stage1_train_models.csv
Stage 1 val file: E:\DEPI-Grad\DEPI-GRADUATION-PROJECT\data\processed\stage1\stage1_val_models.csv
File TRAIN exists: True
File VAL exists: True


In [ ]:
if not STAGE1_TRAIN_PATH.exists():
    raise FileNotFoundError(
        f"Stage 1 TRAIN file not found: {STAGE1_TRAIN_PATH.resolve()}"
    )
if not STAGE1_VAL_PATH.exists():
    raise FileNotFoundError(
        f"Stage 1 VAL file not found: {STAGE1_VAL_PATH.resolve()}"
    )

stage1_train_df = pd.read_csv(STAGE1_TRAIN_PATH)
stage1_val_df = pd.read_csv(STAGE1_VAL_PATH)


print("Shape:", stage1_train_df.shape)
print("Columns:", stage1_train_df.columns.tolist())

print("Shape:", stage1_val_df.shape)
print("Columns:", stage1_val_df.columns.tolist())

Shape: (5165, 56)
Columns: ['title', 'price', 'location', 'ad_url', 'ad_id', 'run_id', 'brand', 'storage_gb', 'ram_gb', 'storage_src', 'condition', 'condition_src', 'ram_src', 'has_5g', 'has_warranty', 'has_box', 'mentions_exchange', 'is_wanted_ad', 'is_accessory_hint', 'is_tablet_hint', 'is_parts_hint', 'accessory_first', 'bundle_hint', 'is_feature_phone_hint', 'price_placeholder', 'loc_area', 'loc_city', 'price_missing', 'price_suspicious', 'price_very_high', 'title_normalized', 'title_punct_normalized', 'title_group', 'iphone_model', 'samsung_model', 'is_phone_listing', 'xiaomi_model', 'oppo_model', 'honor_model', 'realme_model', 'infinix_model', 'vivo_model', 'huawei_model', 'motorola_model', 'oneplus_model', 'zte_model', 'tecno_model', 'itel_model', 'google_model', 'sony_model', 'nokia_model', 'lenovo_model', 'lg_model', 'model_key', 'match_rule', 'confidence']
Shape: (1252, 56)
Columns: ['title', 'price', 'location', 'ad_url', 'ad_id', 'run_id', 'brand', 'storage_gb', 'ram_gb', '

In [ ]:
stage2_train_df = stage1_train_df.copy(deep=True)
stage2_val_df = stage1_val_df.copy(deep=True)



In [ ]:
# Both datasets are processed with exactly the same code
stage2_dfs = {"train": stage2_train_df, "val": stage2_val_df}

# Inspect model keys by brand (train + val)
for name, df in stage2_dfs.items():
    print(f"\n################ {name.upper()} ################")
    for brand in df["brand"].dropna().unique():
        print(f"\n===== {brand.upper()} =====")
        models = (
            df.loc[df["brand"] == brand, "model_key"]
            .dropna()
            .unique()
        )
        print(models[:40])


################ TRAIN ################

===== APPLE =====
['apple iphone 13' 'apple iphone 15 pro' 'apple iphone 11'
 'apple iphone 15 pro max' 'apple iphone 16e' 'apple iphone 17 pro max'
 'apple iphone 14' 'apple iphone 16 pro max' 'apple iphone 13 pro'
 'apple iphone 11 pro max' 'apple iphone 16 pro' 'apple iphone 12'
 'apple iphone x' 'apple iphone 17' 'apple iphone 14 pro max'
 'apple iphone 12 pro' 'apple iphone xs max' 'apple iphone 7 plus'
 'apple iphone 12 mini' 'apple iphone 11 pro' 'apple iphone 17 pro'
 'apple iphone 18 pro' 'apple iphone 16 plus' 'apple iphone 16'
 'apple iphone xr' 'apple iphone 8 plus' 'apple iphone 13 mini'
 'apple iphone 7' 'apple iphone xs' 'apple iphone 14 pro'
 'apple iphone 18 pro max' 'apple iphone 13 pro max'
 'apple iphone 12 pro max' 'apple iphone 6s plus' 'unknown'
 'apple iphone 15' 'apple iphone 6' 'apple iphone 17 air'
 'apple iphone 15 plus' 'apple iphone 17 mini']

===== HONOR =====
['honor magic v5' 'honor robot' 'honor 600' 'honor x8c

In [ ]:

def extract_family(model_key, brand):

    # Handle missing or unknown values
    if pd.isna(model_key) or str(model_key).lower().strip() == "unknown":
        return "unknown"

    if pd.isna(brand):
        return "unknown"

    brand = str(brand).lower().strip()
    model = str(model_key).lower().strip()

    # Remove brand name from model_key
    model = model.removeprefix(brand).strip()

    if not model or model == "unknown":
        return "unknown"

    # Apple
    if brand == "apple":
        return "iphone"

    # Samsung
    elif brand == "samsung":
        if model.startswith("s"):
            return "galaxy s"
        elif model.startswith("a"):
            return "galaxy a"
        elif model.startswith("z trifold"):
            return "galaxy z trifold"
        elif model.startswith("z fold"):
            return "galaxy z fold"
        elif model.startswith("z flip"):
            return "galaxy z flip"
        elif model.startswith("note"):
            return "galaxy note"
        elif model.startswith("j"):
            return "galaxy j"
        elif model.startswith("grand"):
            return "galaxy grand"
        elif model.startswith("mega"):
            return "galaxy mega"
        elif model.startswith("m"):
            return "galaxy m"
        elif model.startswith("xcover"):
            return "galaxy xcover"

    # Xiaomi
    elif brand == "xiaomi":
        if model.startswith("redmi note"):
            return "redmi note"
        elif model.startswith("redmi k"):
            return "redmi k"
        elif model.startswith("redmi"):
            return "redmi"
        elif model.startswith("poco"):
            return "poco"
        else:
            return "xiaomi"

    # Oppo
    elif brand == "oppo":
        for family in ["reno", "find", "a", "f", "r"]:
            if model.startswith(family):
                return family

    # Google
    elif brand == "google":
        return "pixel"

    # Honor
    elif brand == "honor":
        for family in ["magic", "play", "robot", "win", "x", "8c"]:
            if model.startswith(family):
                return family

        if model.split()[0].isdigit():
            return "honor numeric"

    # Vivo
    elif brand == "vivo":
        for family in ["x fold", "iqoo", "x", "y", "v", "s"]:
            if model.startswith(family):
                return family

    # Realme
    elif brand == "realme":
        for family in ["gt", "narzo", "note", "a", "v", "c", "p"]:
            if model.startswith(family):
                return family

        if model[0].isdigit():
            return "realme numeric"

    # Infinix
    elif brand == "infinix":
        for family in ["smart", "note", "hot", "gt", "s"]:
            if model.startswith(family):
                return family

    # Tecno
    elif brand == "tecno":
        for family in ["spark", "camon", "pova", "pop"]:
            if model.startswith(family):
                return family

    # OnePlus
    elif brand == "oneplus":
        for family in ["nord", "ace"]:
            if model.startswith(family):
                return family

        return "oneplus"

    # Huawei
    elif brand == "huawei":
        for family in ["mate", "nova", "pura", "p", "y"]:
            if model.startswith(family):
                return family

    # Motorola
    elif brand == "motorola":
        for family in ["razr", "signature", "stylus", "edge"]:
            if model.startswith(family):
                return family

        if model.startswith("moto g") or model.startswith("g"):
            return "moto g"

    # ZTE
    elif brand == "zte":
        for family in ["red magic", "nubia", "blade"]:
            if model.startswith(family):
                return family

    # Sony
    elif brand == "sony":
        if model.startswith("xperia"):
            return "xperia"
        elif model.startswith("st"):
            return "st"

    # Itel
    elif brand == "itel":
        if model.startswith("a"):
            return "a"
        elif model.startswith("rs"):
            return "rs"

        return "other"

    # Lenovo
    elif brand == "lenovo":
        for family in ["note", "k", "a"]:
            if model.startswith(family):
                return family

    # Nokia
    elif brand == "nokia":
        for family in ["x", "n"]:
            if model.startswith(family):
                return family

    # LG
    elif brand == "lg":
        if model.startswith("g"):
            return "g"

        return "other"

    return "other"


In [ ]:
for name, df in stage2_dfs.items():
    df["family"] = df.apply(
        lambda row: extract_family(row["model_key"], row["brand"]),
        axis=1
    )
    print(f"\n===== {name.upper()} =====")
    print(df[["brand", "model_key", "family"]].head(30))


===== TRAIN =====
      brand                  model_key            family
0     apple            apple iphone 13            iphone
1     honor             honor magic v5             magic
2      oppo           oppo find x9 pro              find
3     honor                honor robot             robot
4    xiaomi            xiaomi 17 ultra            xiaomi
5     apple        apple iphone 15 pro            iphone
6     apple            apple iphone 11            iphone
7     apple    apple iphone 15 pro max            iphone
8    huawei          huawei nova 15max              nova
9     apple           apple iphone 16e            iphone
10     oppo               oppo find x5              find
11  samsung          samsung s24 ultra          galaxy s
12    apple    apple iphone 17 pro max            iphone
13    honor             honor magic v5             magic
14  samsung          samsung s25 ultra          galaxy s
15      NaN                    unknown           unknown
16    apple 

In [ ]:
for name, df in stage2_dfs.items():
    print(f"\n===== {name.upper()} =====")
    print(df["family"].value_counts())


===== TRAIN =====
family
iphone              2435
unknown              470
galaxy a             426
galaxy s             203
reno                 161
                    ... 
galaxy z trifold       1
galaxy mega            1
n                      1
blade                  1
pop                    1
Name: count, Length: 64, dtype: int64

===== VAL =====
family
iphone              455
galaxy a            144
galaxy s            136
unknown             126
reno                 42
xiaomi               40
redmi note           36
x                    29
a                    20
poco                 19
c                    18
realme numeric       16
y                    15
galaxy note          15
honor numeric        13
v                    13
redmi                12
nova                 10
magic                 9
galaxy z flip         8
note                  7
galaxy z fold         6
hot                   6
smart                 6
galaxy m              5
oneplus               5
gt           

In [ ]:
for name, df in stage2_dfs.items():
    print(f"\n===== {name.upper()} =====")
    print("Other count:", (df["family"] == "other").sum())

    print("\nRemaining other models:")
    print(
        df.loc[
            df["family"] == "other",
            ["brand", "model_key"]
        ].drop_duplicates().to_string(index=False)
    )


===== TRAIN =====
Other count: 12

Remaining other models:
  brand             model_key
samsung           samsung f70
  nokia        nokia c21 plus
  nokia             nokia 7.1
infinix       infinix zero 8i
  nokia         nokia g300 5g
 lenovo     lenovo legion y70
   oppo oppo k13 turbo pro 5g
   sony            sony k610i
  honor              honor gt
   itel              itel s24

===== VAL =====
Other count: 1

Remaining other models:
brand      model_key
 itel itel s25 ultra


In [ ]:
for name, df in stage2_dfs.items():
    print(f"\n===== {name.upper()} =====")
    print(df["family"].value_counts())


===== TRAIN =====
family
iphone              2435
unknown              470
galaxy a             426
galaxy s             203
reno                 161
                    ... 
galaxy z trifold       1
galaxy mega            1
n                      1
blade                  1
pop                    1
Name: count, Length: 64, dtype: int64

===== VAL =====
family
iphone              455
galaxy a            144
galaxy s            136
unknown             126
reno                 42
xiaomi               40
redmi note           36
x                    29
a                    20
poco                 19
c                    18
realme numeric       16
y                    15
galaxy note          15
honor numeric        13
v                    13
redmi                12
nova                 10
magic                 9
galaxy z flip         8
note                  7
galaxy z fold         6
hot                   6
smart                 6
galaxy m              5
oneplus               5
gt           

In [ ]:
for name, df in stage2_dfs.items():
    family_counts = df["family"].value_counts()
    print(f"\n===== {name.upper()} =====")
    print(family_counts[family_counts <= 5])


===== TRAIN =====
family
xperia              4
ace                 4
razr                3
red magic           3
robot               3
edge                3
redmi k             3
camon               3
k                   2
pova                2
galaxy grand        2
g                   2
x fold              2
stylus              2
play                2
nord                2
galaxy xcover       2
iqoo                2
signature           2
galaxy z trifold    1
galaxy mega         1
n                   1
blade               1
pop                 1
Name: count, dtype: int64

===== VAL =====
family
galaxy m            5
oneplus             5
gt                  5
mate                4
xperia              3
nord                2
win                 2
edge                2
pura                2
k                   2
find                2
s                   2
galaxy z trifold    1
8c                  1
other               1
rs                  1
p                   1
camon               1


In [ ]:
for name, df in stage2_dfs.items():
    small_families = (
        df.groupby(["brand", "family"])
        .size()
        .reset_index(name="count")
    )
    print(f"\n===== {name.upper()} =====")
    print(
        small_families[small_families["count"] <= 5]
        .sort_values(["brand", "count", "family"])
        .to_string(index=False)
    )


===== TRAIN =====
   brand           family  count
   honor            other      2
   honor             play      2
   honor            robot      3
 infinix            other      1
 infinix                s      1
    itel            other      1
    itel          unknown      2
  lenovo                a      1
  lenovo             note      1
  lenovo            other      1
  lenovo          unknown      1
  lenovo                k      2
      lg          unknown      1
      lg                g      2
motorola        signature      2
motorola           stylus      2
motorola             edge      3
motorola             razr      3
motorola          unknown      3
   nokia                n      1
   nokia          unknown      2
   nokia                x      2
   nokia            other      3
 oneplus             nord      2
 oneplus              ace      4
    oppo            other      2
  realme                a      1
  realme                v      2
  realme          unknow

In [ ]:
for name, df in stage2_dfs.items():
    unknown_models = (
        df.loc[
            df["family"] == "unknown",
            ["brand", "model_key"]
        ]
        .value_counts()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
    )
    print(f"\n===== {name.upper()} =====")
    print(unknown_models.head(30).to_string(index=False))


===== TRAIN =====
   brand   model_key  count
    oppo     unknown     41
  xiaomi     unknown     28
   apple     unknown     27
  huawei     unknown     26
 infinix     unknown     20
   honor     unknown     15
 samsung     unknown     13
    vivo     unknown      5
  realme     unknown      3
motorola     unknown      3
   tecno     unknown      2
    sony     unknown      2
    itel     unknown      2
   nokia     unknown      2
     zte     unknown      2
  lenovo     unknown      1
      lg     unknown      1
     zte zte unknown      1

===== VAL =====
   brand model_key  count
   apple   unknown     19
  realme   unknown     16
 samsung   unknown      7
 infinix   unknown      4
motorola   unknown      2
    oppo   unknown      2
   nokia   unknown      1
    sony   unknown      1
    vivo   unknown      1
  xiaomi   unknown      1
     zte   unknown      1


In [ ]:
for name, df in stage2_dfs.items():
    print(f"\n===== {name.upper()} =====")
    print("Missing family values:", df["family"].isna().sum())
    print("Total rows:", len(df))
    print("Unique families:", df["family"].nunique())


===== TRAIN =====
Missing family values: 0
Total rows: 5165
Unique families: 64

===== VAL =====
Missing family values: 0
Total rows: 1252
Unique families: 51


In [ ]:
for name, df in stage2_dfs.items():
    print(f"\n===== {name.upper()} =====")
    print(
        df.groupby(["brand", "family"])["model_key"]
        .nunique()
        .sort_values(ascending=False)
        .to_string()
    )


===== TRAIN =====
brand     family          
samsung   galaxy a            58
apple     iphone              51
oppo      reno                43
          a                   40
xiaomi    xiaomi              33
samsung   galaxy s            31
xiaomi    redmi note          30
          redmi               28
realme    realme numeric      26
vivo      y                   21
realme    c                   21
honor     x                   21
infinix   hot                 19
xiaomi    poco                18
honor     honor numeric       14
vivo      v                   14
huawei    y                   13
honor     magic               12
huawei    nova                11
          mate                10
samsung   galaxy m            10
          galaxy note          9
          galaxy j             9
oneplus   oneplus              9
samsung   galaxy z fold        9
infinix   note                 8
google    pixel                8
oppo      find                 8
huawei    p                   

In [ ]:
def show_brand_families(brand):
    """model_key -> family mapping for one brand, for train and val together.
    The 'split' column tells whether the model appears in train, val or both."""
    tmp = pd.concat(
        [
            df.loc[df["brand"] == brand, ["model_key", "family"]].assign(split=name)
            for name, df in stage2_dfs.items()
        ],
        ignore_index=True,
    )
    out = (
        tmp.groupby(["model_key", "family"], dropna=False)["split"]
        .agg(lambda s: "+".join(sorted(set(s))))
        .reset_index()
        .sort_values("model_key")
    )
    print(out.to_string(index=False))

show_brand_families("samsung")

              model_key           family     split
            samsung a01         galaxy a train+val
            samsung a02         galaxy a     train
           samsung a02s         galaxy a     train
            samsung a03         galaxy a     train
       samsung a03 core         galaxy a     train
           samsung a03s         galaxy a     train
           samsung a04e         galaxy a train+val
           samsung a04s         galaxy a     train
            samsung a05         galaxy a     train
           samsung a05s         galaxy a train+val
            samsung a06         galaxy a     train
            samsung a07         galaxy a train+val
            samsung a10         galaxy a     train
            samsung a11         galaxy a     train
            samsung a12         galaxy a     train
            samsung a13         galaxy a     train
       samsung a13 plus         galaxy a     train
            samsung a14         galaxy a     train
            samsung a15        

In [ ]:
show_brand_families("xiaomi")

                       model_key     family     split
                         unknown    unknown train+val
                       xiaomi 11     xiaomi     train
                  xiaomi 11 lite     xiaomi train+val
                 xiaomi 11 ultra     xiaomi     train
                  xiaomi 11t pro     xiaomi       val
                       xiaomi 12     xiaomi     train
                   xiaomi 12 pro     xiaomi       val
                       xiaomi 13     xiaomi     train
                  xiaomi 13 lite     xiaomi train+val
                 xiaomi 13 ultra     xiaomi     train
                      xiaomi 13t     xiaomi     train
                       xiaomi 14     xiaomi train+val
                   xiaomi 14 pro     xiaomi train+val
                  xiaomi 14t pro     xiaomi     train
                       xiaomi 15     xiaomi train+val
                   xiaomi 15 pro     xiaomi     train
                 xiaomi 15 ultra     xiaomi     train
                      xiaomi

In [ ]:
show_brand_families("apple")

              model_key  family     split
        apple iphone 11  iphone train+val
    apple iphone 11 pro  iphone train+val
apple iphone 11 pro max  iphone train+val
        apple iphone 12  iphone train+val
   apple iphone 12 mini  iphone train+val
    apple iphone 12 pro  iphone train+val
apple iphone 12 pro max  iphone train+val
        apple iphone 13  iphone train+val
   apple iphone 13 mini  iphone train+val
   apple iphone 13 plus  iphone     train
    apple iphone 13 pro  iphone train+val
apple iphone 13 pro max  iphone train+val
        apple iphone 14  iphone train+val
    apple iphone 14 max  iphone     train
   apple iphone 14 plus  iphone train+val
    apple iphone 14 pro  iphone train+val
apple iphone 14 pro max  iphone train+val
        apple iphone 15  iphone train+val
   apple iphone 15 plus  iphone train+val
    apple iphone 15 pro  iphone train+val
apple iphone 15 pro max  iphone train+val
        apple iphone 16  iphone train+val
   apple iphone 16 plus  iphone tr

In [ ]:
show_brand_families("oppo")

            model_key  family     split
             oppo a12       a     train
             oppo a15       a     train
             oppo a16       a train+val
            oppo a16k       a train+val
       oppo a16pro 5g       a     train
             oppo a17       a     train
             oppo a18       a     train
             oppo a1k       a train+val
              oppo a3       a train+val
             oppo a31       a     train
             oppo a3s       a     train
             oppo a3x       a     train
              oppo a5       a train+val
             oppo a52       a     train
             oppo a53       a     train
             oppo a54       a     train
             oppo a55       a     train
             oppo a57       a       val
             oppo a58       a train+val
             oppo a5i       a     train
           oppo a5pro       a     train
             oppo a5s       a     train
             oppo a5x       a     train
              oppo a6       a     train


In [ ]:
show_brand_families("vivo")

           model_key  family     split
             unknown unknown train+val
vivo iqoo neo 10 pro    iqoo     train
 vivo iqoo neo 7 pro    iqoo     train
            vivo s19       s     train
        vivo s19 pro       s     train
            vivo s20       s     train
        vivo s30 pro       s     train
            vivo v23       v     train
           vivo v23e       v     train
           vivo v27e       v       val
        vivo v29 pro       v     train
           vivo v29e       v     train
            vivo v30       v     train
        vivo v30 pro       v       val
            vivo v40       v train+val
       vivo v40 lite       v     train
            vivo v50       v     train
       vivo v50 lite       v     train
    vivo v50 lite 5g       v     train
            vivo v60       v train+val
       vivo v60 lite       v     train
    vivo v60 lite 5g       v train+val
            vivo v70       v     train
       vivo x fold 3  x fold     train
   vivo x fold 3 pro  x f

In [ ]:
show_brand_families("huawei")

               model_key  family     split
     huawei mate 10 lite    mate train+val
      huawei mate 10 pro    mate     train
         huawei mate 2 x    mate     train
      huawei mate 20 pro    mate     train
      huawei mate 40 pro    mate     train
      huawei mate 50 pro    mate     train
      huawei mate 70 pro    mate     train
           huawei mate 8    mate     train
      huawei mate 80 pro    mate train+val
          huawei mate x5    mate       val
          huawei mate x6    mate     train
huawei mate xt2 ultimate    mate       val
         huawei nova 11i    nova     train
         huawei nova 12i    nova     train
          huawei nova 14    nova     train
       huawei nova 15max    nova     train
       huawei nova 15pro    nova     train
          huawei nova 3i    nova       val
           huawei nova 5    nova train+val
           huawei nova 7    nova train+val
          huawei nova 7i    nova     train
         huawei nova 7se    nova     train
           

In [ ]:
show_brand_families("honor")

         model_key        family     split
          honor 10 honor numeric train+val
     honor 10 lite honor numeric       val
      honor 20 pro honor numeric     train
         honor 200 honor numeric     train
     honor 200 pro honor numeric     train
          honor 30 honor numeric     train
         honor 400 honor numeric train+val
    honor 400 lite honor numeric     train
     honor 400 pro honor numeric train+val
          honor 50 honor numeric       val
         honor 500 honor numeric     train
     honor 500 pro honor numeric       val
         honor 600 honor numeric train+val
    honor 600 lite honor numeric       val
     honor 600 pro honor numeric     train
          honor 70 honor numeric     train
       honor 8 pro honor numeric     train
          honor 8c            8c       val
          honor 90 honor numeric train+val
          honor gt         other     train
 honor magic 4 pro         magic     train
honor magic 5 lite         magic     train
 honor magi

In [ ]:
show_brand_families("motorola")

              model_key    family     split
     motorola edge 2023      edge     train
motorola edge 50 fusion      edge train+val
motorola edge 60 fusion      edge     train
           motorola g04    moto g       val
           motorola g24    moto g     train
      motorola moto g15    moto g     train
      motorola moto g35    moto g     train
  motorola moto g5 plus    moto g     train
      motorola moto g54    moto g     train
      motorola moto g56    moto g     train
      motorola moto g85    moto g     train
       motorola razr 40      razr     train
 motorola razr 50 ultra      razr     train
 motorola razr 60 ultra      razr     train
     motorola signature signature     train
     motorola stylus 5g    stylus     train
                unknown   unknown train+val


In [ ]:
show_brand_families("infinix")

              model_key  family     split
      infinix gt 10 pro      gt     train
      infinix gt 20 pro      gt train+val
      infinix gt 30 pro      gt train+val
      infinix gt 50 pro      gt     train
         infinix hot 10     hot     train
        infinix hot 10s     hot     train
         infinix hot 11     hot     train
    infinix hot 11 play     hot     train
        infinix hot 11s     hot     train
         infinix hot 30     hot     train
        infinix hot 40i     hot     train
     infinix hot 5 lite     hot     train
         infinix hot 50     hot train+val
        infinix hot 50i     hot     train
        infinix hot 50s     hot     train
          infinix hot 6     hot       val
      infinix hot 6 pro     hot     train
         infinix hot 60     hot     train
     infinix hot 60 pro     hot     train
infinix hot 60 pro plus     hot train+val
        infinix hot 60i     hot train+val
         infinix hot 70     hot train+val
          infinix hot 9     hot   

In [ ]:
show_brand_families("tecno")

              model_key  family split
         tecno camon 30   camon train
         tecno camon 40   camon   val
  tecno camon 40 pro 5g   camon train
         tecno camon cm   camon train
            tecno pop 7     pop train
       tecno pova 6 pro    pova train
           tecno pova 7    pova train
tecno spark 20 pro plus   spark train
         tecno spark 30   spark   val
        tecno spark 30c   spark train
     tecno spark 40 pro   spark train
         tecno spark cm   spark train
       tecno spark go 3   spark train
                unknown unknown train


In [ ]:
show_brand_families("realme")

               model_key         family     split
               realme 10 realme numeric     train
           realme 10 pro realme numeric     train
               realme 11 realme numeric train+val
           realme 11 pro realme numeric     train
      realme 11 pro plus realme numeric     train
               realme 12 realme numeric     train
           realme 12 pro realme numeric     train
      realme 12 pro plus realme numeric     train
           realme 13 pro realme numeric     train
               realme 14 realme numeric train+val
           realme 14 pro realme numeric     train
      realme 14 pro plus realme numeric train+val
              realme 14t realme numeric       val
               realme 15 realme numeric train+val
           realme 15 pro realme numeric train+val
      realme 15 pro plus realme numeric     train
              realme 15t realme numeric     train
           realme 16 pro realme numeric     train
      realme 16 pro plus realme numeric train+val


In [ ]:
show_brand_families("oneplus")

             model_key  family     split
        oneplus 10 pro oneplus train+val
            oneplus 11 oneplus     train
           oneplus 12r oneplus     train
           oneplus 13s oneplus     train
           oneplus 13t oneplus     train
            oneplus 15 oneplus       val
             oneplus 5 oneplus     train
oneplus 7t pro mclaren oneplus     train
             oneplus 8 oneplus     train
         oneplus ace 5     ace     train
         oneplus ace 6     ace     train
        oneplus ace 6t     ace     train
        oneplus nord 4    nord     train
      oneplus nord ce4    nord       val
          oneplus open oneplus     train


In [ ]:
show_brand_families("zte")

           model_key    family     split
             unknown   unknown train+val
    zte blade a76 5g     blade     train
       zte nubia a36     nubia     train
    zte nubia a76 5g     nubia     train
     zte nubia neo 2     nubia     train
       zte nubia v60     nubia     train
   zte nubia z30 pro     nubia     train
zte red magic 10 air red magic     train
zte red magic 10 pro red magic     train
     zte red magic 7 red magic     train
         zte unknown   unknown     train


In [ ]:
show_brand_families("google")

               model_key family split
      google pixel 10 5g  pixel train
google pixel 11 pro fold  pixel train
      google pixel 6 pro  pixel train
         google pixel 6a  pixel train
          google pixel 7  pixel train
      google pixel 7 pro  pixel train
         google pixel 8a  pixel train
      google pixel 9 pro  pixel   val
         google pixel 9a  pixel train


In [ ]:
show_brand_families("sony")

           model_key  family     split
          sony k610i   other     train
sony xperia 1 mark 2  xperia     train
sony xperia 1 mark 7  xperia     train
     sony xperia 1 v  xperia       val
sony xperia 4 mark 5  xperia     train
    sony xperia 5 iv  xperia       val
   sony xperia pro-i  xperia       val
 sony xperia z ultra  xperia     train
             unknown unknown train+val


In [ ]:
show_brand_families("nokia")

     model_key  family     split
     nokia 7.1   other     train
nokia c21 plus   other     train
 nokia g300 5g   other     train
      nokia n9       n     train
      nokia x2       x     train
    nokia x350       x     train
       unknown unknown train+val


In [ ]:
for name, df in stage2_dfs.items():
    print(f"\n===== {name.upper()} =====")
    print("Missing families:", df["family"].isna().sum())
    print("Total rows:", len(df))
    print("Unique families:", df["family"].nunique())
    print("\nFamily counts:")
    print(df["family"].value_counts().to_string())


===== TRAIN =====
Missing families: 0
Total rows: 5165
Unique families: 64

Family counts:
family
iphone              2435
unknown              470
galaxy a             426
galaxy s             203
reno                 161
redmi note           160
a                    131
redmi                105
x                     91
honor numeric         81
realme numeric        71
xiaomi                70
hot                   70
c                     68
y                     67
poco                  60
v                     58
note                  38
magic                 34
galaxy note           31
galaxy z fold         28
gt                    24
galaxy m              24
f                     20
p                     18
nova                  18
find                  17
mate                  15
moto g                13
pixel                 13
smart                 12
other                 12
oneplus               11
s                     10
galaxy j              10
nubia                 10
s

In [ ]:
for name, df in stage2_dfs.items():
    df.loc[
        (df["brand"] == "honor") &
        (df["model_key"] == "honor 8c"),
        "family"
    ] = "honor numeric"

    print(f"\n===== {name.upper()} =====")
    print(
        df.loc[
            df["model_key"] == "honor 8c",
            ["model_key", "family"]
        ].drop_duplicates()
    )


===== TRAIN =====
Empty DataFrame
Columns: [model_key, family]
Index: []

===== VAL =====
    model_key         family
274  honor 8c  honor numeric


In [ ]:
for name, df in stage2_dfs.items():
    print(f"\n===== {name.upper()} =====")
    print("Missing families:", df["family"].isna().sum())
    print("Total rows:", len(df))
    print("Unique families:", df["family"].nunique())
    print("\nRemaining 'other':", (df["family"] == "other").sum())
    print("\nUpdated Honor families:")
    print(
        df.loc[
            df["brand"] == "honor",
            "family"
        ].value_counts().to_string()
    )


===== TRAIN =====
Missing families: 0
Total rows: 5165
Unique families: 64

Remaining 'other': 12

Updated Honor families:
family
honor numeric    81
x                80
magic            34
unknown          15
win               7
robot             3
play              2
other             2

===== VAL =====
Missing families: 0
Total rows: 1252
Unique families: 50

Remaining 'other': 1

Updated Honor families:
family
x                25
honor numeric    14
magic             9
win               2


In [ ]:
print("train:", stage2_train_df.shape)
print("val:  ", stage2_val_df.shape)

train: (5165, 57)
val:   (1252, 57)


In [ ]:
print("train columns:", stage2_train_df.columns.tolist())
print("val columns:  ", stage2_val_df.columns.tolist())


train columns: ['title', 'price', 'location', 'ad_url', 'ad_id', 'run_id', 'brand', 'storage_gb', 'ram_gb', 'storage_src', 'condition', 'condition_src', 'ram_src', 'has_5g', 'has_warranty', 'has_box', 'mentions_exchange', 'is_wanted_ad', 'is_accessory_hint', 'is_tablet_hint', 'is_parts_hint', 'accessory_first', 'bundle_hint', 'is_feature_phone_hint', 'price_placeholder', 'loc_area', 'loc_city', 'price_missing', 'price_suspicious', 'price_very_high', 'title_normalized', 'title_punct_normalized', 'title_group', 'iphone_model', 'samsung_model', 'is_phone_listing', 'xiaomi_model', 'oppo_model', 'honor_model', 'realme_model', 'infinix_model', 'vivo_model', 'huawei_model', 'motorola_model', 'oneplus_model', 'zte_model', 'tecno_model', 'itel_model', 'google_model', 'sony_model', 'nokia_model', 'lenovo_model', 'lg_model', 'model_key', 'match_rule', 'confidence', 'family']
val columns:   ['title', 'price', 'location', 'ad_url', 'ad_id', 'run_id', 'brand', 'storage_gb', 'ram_gb', 'storage_src', 

## Save the family results

Save these files (train + val) for the feature-pipeline notebook that runs next.

In [ ]:
STAGE2_TRAIN_PATH = BASE_DIR / "data" / "processed" / "stage2" / "stage2_train_df.csv"
STAGE2_VAL_PATH   = BASE_DIR / "data" / "processed" / "stage2" / "stage2_val_df.csv"

STAGE2_TRAIN_PATH.parent.mkdir(parents=True, exist_ok=True)

stage2_train_df.to_csv(STAGE2_TRAIN_PATH, index=False)
stage2_val_df.to_csv(STAGE2_VAL_PATH, index=False)

print("Stage 2 TRAIN saved:", STAGE2_TRAIN_PATH.resolve(), stage2_train_df.shape)
print("Stage 2 VAL saved:  ", STAGE2_VAL_PATH.resolve(), stage2_val_df.shape)

Stage 2 TRAIN saved: E:\DEPI-Grad\DEPI-GRADUATION-PROJECT\data\processed\stage2\stage2_train_df.csv (5165, 57)
Stage 2 VAL saved:   E:\DEPI-Grad\DEPI-GRADUATION-PROJECT\data\processed\stage2\stage2_val_df.csv (1252, 57)
